# Medical Conversation Analyzer

Pipeline: Audio → Zipformer ASR → ICA Speaker Diarization → LLM Classification

**Output:** JSON với transcript (bác sĩ/bệnh nhân) + phân loại:
- Quá trình bệnh lý
- Tiền sử bệnh nhân
- Tiền sử gia đình

---
**Runtime:** Chọn GPU (Runtime → Change runtime type → T4 GPU)

## 1. Install Dependencies

In [ ]:
!pip install -q sherpa-onnx numpy scipy scikit-learn httpx soundfile noisereduce python-dotenv huggingface_hub

## 2. Download Zipformer ASR Model (Vietnamese)

In [ ]:
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/content/models/zipformer-offline"
os.makedirs(MODEL_DIR, exist_ok=True)

snapshot_download(
    repo_id="hynt/Zipformer-30M-RNNT-6000h",
    local_dir=MODEL_DIR,
)
print("ASR model downloaded:")
!ls -la {MODEL_DIR}

## 3. Download & Start LLM (Qwen3-4B via llama-server)

Cần GPU T4 trở lên. File GGUF ~2.5GB.

In [ ]:
%%bash
# Download llama.cpp pre-built binary
LLAMA_VERSION="b5600"
wget -q "https://github.com/ggml-org/llama.cpp/releases/download/${LLAMA_VERSION}/llama-${LLAMA_VERSION}-bin-ubuntu-x64-cuda.zip" -O /tmp/llama.zip
unzip -qo /tmp/llama.zip -d /content/llama-cpp
chmod +x /content/llama-cpp/build/bin/llama-server
echo "llama.cpp ready"

In [ ]:
from huggingface_hub import hf_hub_download

GGUF_PATH = hf_hub_download(
    repo_id="unsloth/Qwen3-4B-GGUF",
    filename="Qwen3-4B-Q4_K_M.gguf",
    local_dir="/content/models",
)
print(f"GGUF model: {GGUF_PATH}")

In [ ]:
import subprocess, time

# Start llama-server in background
llm_proc = subprocess.Popen([
    "/content/llama-cpp/build/bin/llama-server",
    "-m", GGUF_PATH,
    "--host", "0.0.0.0", "--port", "8899",
    "-ngl", "99", "-c", "4096",
    "--reasoning-budget", "0", "--jinja",
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to be ready
import httpx
for i in range(30):
    try:
        r = httpx.get("http://127.0.0.1:8899/health", timeout=2)
        if r.status_code == 200:
            print(f"LLM server ready (took {i+1}s)")
            break
    except:
        pass
    time.sleep(1)
else:
    print("WARNING: LLM server did not start in 30s")

## 4. Upload Audio File

Upload file hội thoại bác sĩ - bệnh nhân (.wav, .mp3, .flac)

In [ ]:
from google.colab import files

print("Upload file audio cuộc hội thoại:")
uploaded = files.upload()
AUDIO_FILE = list(uploaded.keys())[0]
print(f"\nFile: {AUDIO_FILE} ({len(uploaded[AUDIO_FILE])/1024:.1f} KB)")

## 5. Run Pipeline

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

In [ ]:
import numpy as np
import soundfile as sf
from scipy.signal import butter, sosfilt, resample
from sklearn.decomposition import FastICA
import sherpa_onnx
import httpx
import json

# === CONFIG ===
ASR_SAMPLE_RATE = 16000
LLM_URL = "http://127.0.0.1:8899/v1"

# === PREPROCESSING ===
def bandpass_filter(audio, sample_rate, low_hz=300, high_hz=3400, order=5):
    nyquist = sample_rate / 2
    low = low_hz / nyquist
    high = min(high_hz / nyquist, 0.99)
    sos = butter(order, [low, high], btype="band", output="sos")
    return sosfilt(sos, audio).astype(np.float32)

def preprocess(audio, sample_rate):
    import noisereduce as nr
    if audio.ndim == 1:
        audio = nr.reduce_noise(y=audio, sr=sample_rate, prop_decrease=0.8).astype(np.float32)
        audio = bandpass_filter(audio, sample_rate)
    else:
        for ch in range(audio.shape[0]):
            audio[ch] = nr.reduce_noise(y=audio[ch], sr=sample_rate, prop_decrease=0.8).astype(np.float32)
            audio[ch] = bandpass_filter(audio[ch], sample_rate)
    return audio

# === SPEAKER DIARIZATION (ICA) ===
def ica_separate(audio, n_components=2):
    """FastICA for multi-channel audio."""
    ica = FastICA(n_components=n_components, random_state=42, max_iter=500)
    sources = ica.fit_transform(audio.T).T
    for i in range(sources.shape[0]):
        peak = np.max(np.abs(sources[i]))
        if peak > 0:
            sources[i] /= peak
    return sources.astype(np.float32)

def segment_by_energy(audio, sample_rate, frame_ms=30, min_segment_ms=500):
    """Energy-based segmentation for mono audio."""
    frame_size = int(sample_rate * frame_ms / 1000)
    n_frames = len(audio) // frame_size
    if n_frames == 0:
        return []

    energies = np.array([
        np.sqrt(np.mean(audio[i*frame_size:(i+1)*frame_size]**2))
        for i in range(n_frames)
    ])
    silence_threshold = max(np.mean(energies) * 0.15, 0.001)
    is_speech = energies > silence_threshold

    segments = []
    in_speech = False
    seg_start = 0
    speaker = 0
    min_frames = max(1, int(min_segment_ms / frame_ms))

    for i in range(n_frames):
        if is_speech[i] and not in_speech:
            seg_start = i
            in_speech = True
        elif not is_speech[i] and in_speech:
            if (i - seg_start) >= min_frames:
                segments.append({
                    "speaker": speaker,
                    "audio": audio[seg_start*frame_size:i*frame_size],
                })
                speaker = 1 - speaker
            in_speech = False

    if in_speech and (n_frames - seg_start) >= min_frames:
        segments.append({"speaker": speaker, "audio": audio[seg_start*frame_size:]})

    return segments

def segment_ica_sources(sources, sample_rate, frame_ms=30, threshold=0.03, min_segment_ms=300):
    """Segment ICA-separated sources by dominant energy."""
    frame_size = int(sample_rate * frame_ms / 1000)
    n_frames = sources.shape[1] // frame_size
    n_sources = sources.shape[0]

    energies = np.zeros((n_sources, n_frames))
    for s in range(n_sources):
        for i in range(n_frames):
            chunk = sources[s, i*frame_size:(i+1)*frame_size]
            energies[s, i] = np.sqrt(np.mean(chunk**2))

    dominant = np.argmax(energies, axis=0)
    dominant[np.max(energies, axis=0) < threshold] = -1

    segments = []
    min_frames = max(1, int(min_segment_ms / frame_ms))
    seg_start = 0
    current = dominant[0]

    for i in range(1, n_frames):
        if dominant[i] != current:
            if current >= 0 and (i - seg_start) >= min_frames:
                spk = int(current)
                segments.append({
                    "speaker": spk,
                    "audio": sources[spk, seg_start*frame_size:i*frame_size],
                })
            seg_start = i
            current = dominant[i]

    if current >= 0 and (n_frames - seg_start) >= min_frames:
        spk = int(current)
        segments.append({"speaker": spk, "audio": sources[spk, seg_start*frame_size:]})

    return segments

def diarize(audio, sample_rate):
    """Main diarization: ICA for stereo, energy-based for mono."""
    if audio.ndim == 2 and audio.shape[0] >= 2:
        print(f"  Multi-channel ({audio.shape[0]}ch) → ICA separation")
        sources = ica_separate(audio, n_components=2)
        return segment_ica_sources(sources, sample_rate)
    else:
        if audio.ndim == 2:
            audio = audio[0]
        print(f"  Mono → energy-based segmentation")
        return segment_by_energy(audio, sample_rate)

# === ASR (Zipformer) ===
print("Loading Zipformer ASR...")
recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=f"{MODEL_DIR}/encoder-epoch-20-avg-10.int8.onnx",
    decoder=f"{MODEL_DIR}/decoder-epoch-20-avg-10.int8.onnx",
    joiner=f"{MODEL_DIR}/joiner-epoch-20-avg-10.int8.onnx",
    tokens=f"{MODEL_DIR}/config.json",
    num_threads=2,
    sample_rate=ASR_SAMPLE_RATE,
    feature_dim=80,
    decoding_method="greedy_search",
)
print("ASR ready.")

def transcribe(audio_segment, sample_rate):
    if len(audio_segment) == 0:
        return ""
    stream = recognizer.create_stream()
    stream.accept_waveform(sample_rate, audio_segment)
    recognizer.decode_stream(stream)
    return (stream.result.text or "").strip()

# === LLM ===
def llm_complete(system_prompt, user_content, temperature=0.1):
    resp = httpx.post(
        f"{LLM_URL}/chat/completions",
        json={
            "model": "qwen",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            "temperature": temperature,
            "max_tokens": 2048,
            "stream": False,
        },
        timeout=120.0,
    )
    return resp.json()["choices"][0]["message"]["content"]

print("\nAll components loaded.")

In [ ]:
# === RUN FULL PIPELINE ===
print(f"=== Analyzing: {AUDIO_FILE} ===")

# 1. Load audio
audio, sr = sf.read(AUDIO_FILE, dtype="float32")
if audio.ndim == 2:
    audio = audio.T  # (channels, samples)
if sr != ASR_SAMPLE_RATE:
    if audio.ndim == 1:
        n = int(len(audio) * ASR_SAMPLE_RATE / sr)
        audio = resample(audio, n).astype(np.float32)
    else:
        n = int(audio.shape[1] * ASR_SAMPLE_RATE / sr)
        audio = np.array([resample(audio[ch], n).astype(np.float32) for ch in range(audio.shape[0])])
    sr = ASR_SAMPLE_RATE
print(f"1. Audio loaded: shape={audio.shape}, sr={sr}")

# 2. Preprocess
audio = preprocess(audio, sr)
print("2. Preprocessing done (noise reduction + bandpass)")

# 3. Speaker Diarization
print("3. Speaker diarization:")
segments = diarize(audio, sr)
print(f"   → {len(segments)} segments found")

# 4. ASR each segment
print("4. ASR (Zipformer):")
for seg in segments:
    seg["text"] = transcribe(seg["audio"], sr)
segments = [s for s in segments if s["text"].strip()]
print(f"   → {len(segments)} non-empty segments")
for i, s in enumerate(segments[:5]):
    print(f"   [{i}] speaker={s['speaker']}: {s['text'][:60]}")

# 5. LLM spell correction
print("5. LLM spell correction...")
numbered = "\n".join(f"{i+1}. {s['text']}" for i, s in enumerate(segments))
corrected_raw = llm_complete(
    "Sửa chính tả tiếng Việt cho các câu y khoa sau. Chỉ sửa lỗi chính tả, KHÔNG thay đổi nội dung. "
    "Trả về đúng số dòng, mỗi dòng bắt đầu bằng số thứ tự. Ví dụ:\n1. câu đã sửa\n2. câu đã sửa",
    numbered,
)
corrected_lines = []
for line in corrected_raw.strip().split("\n"):
    line = line.strip()
    if not line:
        continue
    parts = line.split(". ", 1)
    corrected_lines.append(parts[1] if len(parts) == 2 and parts[0].isdigit() else line)

if len(corrected_lines) == len(segments):
    for i, s in enumerate(segments):
        s["text"] = corrected_lines[i]
print(f"   Done ({len(corrected_lines)} lines)")

# 6. LLM role assignment
print("6. LLM role assignment (bác_sĩ / bệnh_nhân)...")
numbered2 = "\n".join(f"{i+1}. {s['text']}" for i, s in enumerate(segments))
roles_raw = llm_complete(
    'Phân tích đoạn hội thoại y khoa sau. Xác định vai trò của từng câu: '
    '"bác_sĩ" hoặc "bệnh_nhân". Bác sĩ thường hỏi triệu chứng, tiền sử; '
    'bệnh nhân thường mô tả triệu chứng, trả lời câu hỏi.\n\n'
    'Trả về JSON array chứa role cho mỗi câu, ví dụ:\n'
    '["bác_sĩ", "bệnh_nhân", "bác_sĩ", ...]\nChỉ trả JSON, không giải thích.',
    numbered2,
)
roles_raw = roles_raw.strip()
if roles_raw.startswith("```"):
    roles_raw = roles_raw.split("\n", 1)[1].rsplit("```", 1)[0]
try:
    roles = json.loads(roles_raw)
    if len(roles) != len(segments):
        raise ValueError("length mismatch")
except:
    roles = ["bác_sĩ" if i % 2 == 0 else "bệnh_nhân" for i in range(len(segments))]
print(f"   Roles: {roles[:6]}...")

# 7. Build transcript
transcript = [{"role": roles[i], "text": segments[i]["text"]} for i in range(len(segments))]

# 8. LLM classification
print("7. LLM medical classification...")
conv_text = "\n".join(f"[{t['role']}]: {t['text']}" for t in transcript)
classification_raw = llm_complete(
    'Bạn là trợ lý y khoa. Phân tích đoạn hội thoại giữa bác sĩ và bệnh nhân, '
    'trích xuất thông tin vào 3 mục sau. Chỉ trả về JSON, không giải thích.\n\n'
    'Output JSON format:\n'
    '{\n'
    '  "qua_trinh_benh_ly": "mô tả quá trình bệnh lý hiện tại",\n'
    '  "tien_su_benh_nhan": "tiền sử bệnh của bệnh nhân",\n'
    '  "tien_su_gia_dinh": "tiền sử bệnh trong gia đình"\n'
    '}\n\n'
    'Nếu không có thông tin cho mục nào, ghi "Không có thông tin".',
    conv_text,
)
classification_raw = classification_raw.strip()
if classification_raw.startswith("```"):
    classification_raw = classification_raw.split("\n", 1)[1].rsplit("```", 1)[0]
try:
    classification = json.loads(classification_raw)
except:
    classification = {
        "qua_trinh_benh_ly": classification_raw,
        "tien_su_benh_nhan": "Không thể phân tích",
        "tien_su_gia_dinh": "Không thể phân tích",
    }

print("\n=== DONE ===")

## 6. Results

In [ ]:
# Final JSON output
result = {
    "transcript": transcript,
    "qua_trinh_benh_ly": classification.get("qua_trinh_benh_ly", "Không có thông tin"),
    "tien_su_benh_nhan": classification.get("tien_su_benh_nhan", "Không có thông tin"),
    "tien_su_gia_dinh": classification.get("tien_su_gia_dinh", "Không có thông tin"),
}

print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# Display formatted
from IPython.display import Markdown, display

md = "## Transcript\n\n"
for t in transcript:
    icon = "🩺" if t["role"] == "bác_sĩ" else "🧑"
    md += f"{icon} **{t['role']}**: {t['text']}\n\n"

md += "---\n"
md += f"## Quá trình bệnh lý\n{result['qua_trinh_benh_ly']}\n\n"
md += f"## Tiền sử bệnh nhân\n{result['tien_su_benh_nhan']}\n\n"
md += f"## Tiền sử gia đình\n{result['tien_su_gia_dinh']}\n\n"

display(Markdown(md))

In [ ]:
# Save to file & download
output_path = AUDIO_FILE.rsplit(".", 1)[0] + "_result.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Saved: {output_path}")
files.download(output_path)

## Cleanup

In [ ]:
# Stop LLM server
llm_proc.terminate()
print("LLM server stopped.")